# Visualization of Recommendation Results

In [70]:
from neo4j import GraphDatabase
from pyvis.network import Network

In [71]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "12CodeNEO4j") #### HIDE THIS BEFORE PUSH

## Full Graph

In [72]:
# to visualize full graph, not that useful and takes a while
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session() as session:
    # pull all edges from neo4j
    result = session.run("""
        MATCH (a:Song)-[r:SIMILAR_TO]->(b:Song)
        RETURN a.title AS song_a, a.artist AS artist_a,
               b.title AS song_b, b.artist AS artist_b,
               r.similarity AS score
    """)
    rows = result.data()

driver.close()

In [73]:
# # long rendering time
# net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="in_line")

# for row in rows:
#     # color by artist type
#     if "The Strokes" in row["artist_a"]:
#         color_a = "#e84118"   # red = strokes
#     else:
#         color_a = "#ffffff"   # white = other

#     if "The Strokes" in row["artist_b"]:
#         color_b = "#e84118"
#     elif row["artist_b"] in ["Kim Petras", "Celso Blues Boy", "SHY FX;Chronixx;S.P.Y",
#                               "Orchestral Manoeuvres In The Dark", "Firehouse"]:
#         color_b = "#4cd137"   # green = recommendations
#     else:
#         color_b = "#ffffff"

#     net.add_node(row["song_a"], label=row["song_a"], color=color_a, title=row["artist_a"])
#     net.add_node(row["song_b"], label=row["song_b"], color=color_b, title=row["artist_b"])
#     net.add_edge(row["song_a"], row["song_b"], value=row["score"],
#                  title=f"{row['score']:.4f}")
    
# net.show('network_graph.html', notebook=False)

## Largest Connected Component

In [74]:
import networkx as nx

# build networkx graph first
G = nx.Graph()
for row in rows:
    G.add_node(row["song_a"], artist=row["artist_a"])
    G.add_node(row["song_b"], artist=row["artist_b"])
    G.add_edge(row["song_a"], row["song_b"], weight=row["score"])

# keep only largest connected component
largest = max(nx.connected_components(G), key=len)
G = G.subgraph(largest).copy()
print(f"Largest component: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# now build pyvis from filtered graph
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="in_line")

for node, data in G.nodes(data=True):
    if "The Strokes" in data.get("artist", ""):
        color = "#e84118"
    elif data.get("artist", "") in ["Kim Petras", "Celso Blues Boy", "SHY FX;Chronixx;S.P.Y",
                                     "Orchestral Manoeuvres In The Dark", "Firehouse"]:
        color = "#4cd137"
    else:
        color = "#ffffff"
    net.add_node(node, label=node, color=color, title=data.get("artist", ""))

for u, v, data in G.edges(data=True):
    net.add_edge(u, v, value=data["weight"], title=f"{data['weight']:.4f}")

net.show('network_graph.html', notebook=False)

Largest component: 529 nodes, 1468 edges
network_graph.html


## Recommendation Graph Results, with 50 nodes

In [75]:
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session() as session:
    # pull all edges from neo4j
    result = session.run("""
        MATCH (seed:Song)-[r:SIMILAR_TO]-(rec:Song)
        WHERE seed.artist CONTAINS "The Strokes" // REPLACE W DESIRED ARTIST HERE !
        AND NOT rec.artist CONTAINS "The Strokes"
        RETURN seed.title AS seed_title, seed.artist AS seed_artist,
               rec.title AS rec_title, rec.artist AS rec_artist,
               r.similarity AS score
        ORDER BY score DESC
        LIMIT 50;""") ## to show other songs NOT selected
    rows = result.data()

driver.close()

In [76]:
# top 5 rec titles
top5 = [row["rec_title"] for row in rows[:5]]

net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="in_line")

for row in rows:
    rec_color = "#4cd137" if row["rec_title"] in top5 else "#ffffff"  # green = top5, white = rest
    net.add_node(row["seed_title"], label=row["seed_title"], color="#e84118", title=row["seed_artist"])
    net.add_node(row["rec_title"],  label=row["rec_title"],  color=rec_color,  title=row["rec_artist"])
    net.add_edge(row["seed_title"], row["rec_title"], value=row["score"],
                 title=f"{row['score']:.4f}")

net.show('network_graph.html', notebook=False)

network_graph.html
